# 基于滑动窗口的机器阅读理解任务的实现

# Step1 导入相关包

In [1]:
!pip install nltk

In [2]:
import nltk
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
from datasets import load_dataset,DatasetDict
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer, DefaultDataCollator

# Step2 数据集加载

In [4]:
datasets = load_dataset("hfl/cmrc2018")

In [5]:
datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'context', 'question', 'answers'],
        num_rows: 10142
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'answers'],
        num_rows: 3219
    })
    test: Dataset({
        features: ['id', 'context', 'question', 'answers'],
        num_rows: 1002
    })
})

In [6]:
datasets["train"][0]

{'id': 'TRAIN_186_QUERY_0',
 'context': '范廷颂枢机（，），圣名保禄·若瑟（），是越南罗马天主教枢机。1963年被任为主教；1990年被擢升为天主教河内总教区宗座署理；1994年被擢升为总主教，同年年底被擢升为枢机；2009年2月离世。范廷颂于1919年6月15日在越南宁平省天主教发艳教区出生；童年时接受良好教育后，被一位越南神父带到河内继续其学业。范廷颂于1940年在河内大修道院完成神学学业。范廷颂于1949年6月6日在河内的主教座堂晋铎；及后被派到圣女小德兰孤儿院服务。1950年代，范廷颂在河内堂区创建移民接待中心以收容到河内避战的难民。1954年，法越战争结束，越南民主共和国建都河内，当时很多天主教神职人员逃至越南的南方，但范廷颂仍然留在河内。翌年管理圣若望小修院；惟在1960年因捍卫修院的自由、自治及拒绝政府在修院设政治课的要求而被捕。1963年4月5日，教宗任命范廷颂为天主教北宁教区主教，同年8月15日就任；其牧铭为「我信天主的爱」。由于范廷颂被越南政府软禁差不多30年，因此他无法到所属堂区进行牧灵工作而专注研读等工作。范廷颂除了面对战争、贫困、被当局迫害天主教会等问题外，也秘密恢复修院、创建女修会团体等。1990年，教宗若望保禄二世在同年6月18日擢升范廷颂为天主教河内总教区宗座署理以填补该教区总主教的空缺。1994年3月23日，范廷颂被教宗若望保禄二世擢升为天主教河内总教区总主教并兼天主教谅山教区宗座署理；同年11月26日，若望保禄二世擢升范廷颂为枢机。范廷颂在1995年至2001年期间出任天主教越南主教团主席。2003年4月26日，教宗若望保禄二世任命天主教谅山教区兼天主教高平教区吴光杰主教为天主教河内总教区署理主教；及至2005年2月19日，范廷颂因获批辞去总主教职务而荣休；吴光杰同日真除天主教河内总教区总主教职务。范廷颂于2009年2月22日清晨在河内离世，享年89岁；其葬礼于同月26日上午在天主教河内总教区总主教座堂举行。',
 'question': '范廷颂是什么时候被任为主教的？',
 'answers': {'text': ['1963年'], 'answer_start': [30]}}

# Step3 数据预处理

In [7]:
tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-macbert-base")

In [8]:
sample_dataset=datasets["train"].select(range(10))

In [9]:
sample_examples=tokenizer(text=list(sample_dataset["question"]),
             text_pair=list(sample_dataset["context"]),
             max_length=384,truncation="only_second",
             padding="max_length",return_offsets_mapping=True,
             return_overflowing_tokens=True,stride=128)
sample_examples.keys()

KeysView({'input_ids': [[101, 5745, 2455, 7563, 3221, 784, 720, 3198, 952, 6158, 818, 711, 712, 3136, 4638, 8043, 102, 5745, 2455, 7563, 3364, 3322, 8020, 8024, 8021, 8024, 1760, 1399, 924, 4882, 185, 5735, 4449, 8020, 8021, 8024, 3221, 6632, 1298, 5384, 7716, 1921, 712, 3136, 3364, 3322, 511, 9155, 2399, 6158, 818, 711, 712, 3136, 8039, 8431, 2399, 6158, 3091, 1285, 711, 1921, 712, 3136, 3777, 1079, 2600, 3136, 1277, 2134, 2429, 5392, 4415, 8039, 8447, 2399, 6158, 3091, 1285, 711, 2600, 712, 3136, 8024, 1398, 2399, 2399, 2419, 6158, 3091, 1285, 711, 3364, 3322, 8039, 8170, 2399, 123, 3299, 4895, 686, 511, 5745, 2455, 7563, 754, 9915, 2399, 127, 3299, 8115, 3189, 1762, 6632, 1298, 2123, 2398, 4689, 1921, 712, 3136, 1355, 5683, 3136, 1277, 1139, 4495, 8039, 4997, 2399, 3198, 2970, 1358, 5679, 1962, 3136, 5509, 1400, 8024, 6158, 671, 855, 6632, 1298, 4868, 4266, 2372, 1168, 3777, 1079, 5326, 5330, 1071, 2110, 689, 511, 5745, 2455, 7563, 754, 9211, 2399, 1762, 3777, 1079, 1920, 934, 6887,

In [10]:
sample_examples["overflow_to_sample_mapping"],len(sample_examples["overflow_to_sample_mapping"])

([0,
  0,
  0,
  1,
  1,
  1,
  2,
  2,
  2,
  3,
  3,
  3,
  4,
  4,
  4,
  5,
  5,
  5,
  6,
  6,
  6,
  7,
  7,
  7,
  8,
  8,
  8,
  9,
  9],
 29)

In [11]:
for sen in tokenizer.batch_decode(sample_examples["input_ids"][:3]):
  print(sen)


[CLS] 范 廷 颂 是 什 么 时 候 被 任 为 主 教 的 ？ [SEP] 范 廷 颂 枢 机 （ ， ） ， 圣 名 保 禄 · 若 瑟 （ ） ， 是 越 南 罗 马 天 主 教 枢 机 。 1963 年 被 任 为 主 教 ； 1990 年 被 擢 升 为 天 主 教 河 内 总 教 区 宗 座 署 理 ； 1994 年 被 擢 升 为 总 主 教 ， 同 年 年 底 被 擢 升 为 枢 机 ； 2009 年 2 月 离 世 。 范 廷 颂 于 1919 年 6 月 15 日 在 越 南 宁 平 省 天 主 教 发 艳 教 区 出 生 ； 童 年 时 接 受 良 好 教 育 后 ， 被 一 位 越 南 神 父 带 到 河 内 继 续 其 学 业 。 范 廷 颂 于 1940 年 在 河 内 大 修 道 院 完 成 神 学 学 业 。 范 廷 颂 于 1949 年 6 月 6 日 在 河 内 的 主 教 座 堂 晋 铎 ； 及 后 被 派 到 圣 女 小 德 兰 孤 儿 院 服 务 。 1950 年 代 ， 范 廷 颂 在 河 内 堂 区 创 建 移 民 接 待 中 心 以 收 容 到 河 内 避 战 的 难 民 。 1954 年 ， 法 越 战 争 结 束 ， 越 南 民 主 共 和 国 建 都 河 内 ， 当 时 很 多 天 主 教 神 职 人 员 逃 至 越 南 的 南 方 ， 但 范 廷 颂 仍 然 留 在 河 内 。 翌 年 管 理 圣 若 望 小 修 院 ； 惟 在 1960 年 因 捍 卫 修 院 的 自 由 、 自 治 及 拒 绝 政 府 在 修 院 设 政 治 课 的 要 求 而 被 捕 。 1963 年 4 月 5 日 ， 教 宗 任 命 范 廷 颂 为 天 主 教 北 宁 教 区 主 教 ， 同 年 8 月 15 日 就 任 ； 其 牧 铭 为 「 我 信 [SEP]
[CLS] 范 廷 颂 是 什 么 时 候 被 任 为 主 教 的 ？ [SEP] 越 南 民 主 共 和 国 建 都 河 内 ， 当 时 很 多 天 主 教 神 职 人 员 逃 至 越 南 的 南 方 ， 但 范 廷 颂 仍 然 留 在 河 内 。 翌 年 管 理 圣 若 望 小 修 院 ； 惟 在 1960 年 因 捍 卫 修 院 的 自 由 、 自 治 及 拒 

In [12]:
print(sample_examples["offset_mapping"][:3])

[[(0, 0), (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15), (0, 0), (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15), (15, 16), (16, 17), (17, 18), (18, 19), (19, 20), (20, 21), (21, 22), (22, 23), (23, 24), (24, 25), (25, 26), (26, 27), (27, 28), (28, 29), (29, 30), (30, 34), (34, 35), (35, 36), (36, 37), (37, 38), (38, 39), (39, 40), (40, 41), (41, 45), (45, 46), (46, 47), (47, 48), (48, 49), (49, 50), (50, 51), (51, 52), (52, 53), (53, 54), (54, 55), (55, 56), (56, 57), (57, 58), (58, 59), (59, 60), (60, 61), (61, 62), (62, 63), (63, 67), (67, 68), (68, 69), (69, 70), (70, 71), (71, 72), (72, 73), (73, 74), (74, 75), (75, 76), (76, 77), (77, 78), (78, 79), (79, 80), (80, 81), (81, 82), (82, 83), (83, 84), (84, 85), (85, 86), (86, 87), (87, 91), (91, 92), (92, 93), (93, 94), (94, 95), (95, 96), (96, 97), (97, 98), (98, 99), 

In [13]:
print(sample_examples["offset_mapping"][0],len(sample_examples["offset_mapping"][0]))

[(0, 0), (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15), (0, 0), (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15), (15, 16), (16, 17), (17, 18), (18, 19), (19, 20), (20, 21), (21, 22), (22, 23), (23, 24), (24, 25), (25, 26), (26, 27), (27, 28), (28, 29), (29, 30), (30, 34), (34, 35), (35, 36), (36, 37), (37, 38), (38, 39), (39, 40), (40, 41), (41, 45), (45, 46), (46, 47), (47, 48), (48, 49), (49, 50), (50, 51), (51, 52), (52, 53), (53, 54), (54, 55), (55, 56), (56, 57), (57, 58), (58, 59), (59, 60), (60, 61), (61, 62), (62, 63), (63, 67), (67, 68), (68, 69), (69, 70), (70, 71), (71, 72), (72, 73), (73, 74), (74, 75), (75, 76), (76, 77), (77, 78), (78, 79), (79, 80), (80, 81), (81, 82), (82, 83), (83, 84), (84, 85), (85, 86), (86, 87), (87, 91), (91, 92), (92, 93), (93, 94), (94, 95), (95, 96), (96, 97), (97, 98), (98, 99), (

In [14]:
sample_mapping=sample_examples.pop("overflow_to_sample_mapping")

In [15]:
for idx,_ in enumerate(sample_mapping):
  answer=sample_dataset["answers"][sample_mapping[idx]]
  start_char=answer["answer_start"][0]
  end_char=start_char+len(answer["text"][0])
  # 定位答案在token中的起始位置和结束位置
  # 一种策略，我们要拿到context的起始和结束，然后从左右两侧向答案逼近
  context_start=sample_examples.sequence_ids(idx).index(1)
  context_end=sample_examples.sequence_ids(idx).index(None,context_start)-1
  offset=sample_examples.get("offset_mapping")[idx]
  example_ids=[]
  # 判断答案是否在context中
  if offset[context_end][1]<start_char or offset[context_start][0]>end_char:
    start_token_pos=0
    end_token_pos=0
  else:
    token_id=context_start
    while token_id<=context_end and offset[token_id][0]<start_char:
      token_id +=1
    start_token_pos=token_id
    token_id=context_end
    while token_id>=context_start and offset[token_id][1]>end_char:
      token_id -=1
    end_token_pos=token_id

  print(answer,start_char,end_char,context_start,context_end,start_token_pos,end_token_pos)
  print("token answer decode:",tokenizer.decode(sample_examples["input_ids"][idx][start_token_pos:end_token_pos+1]))


{'text': ['1963年'], 'answer_start': [30]} 30 35 17 382 47 48
token answer decode: 1963 年
{'text': ['1963年'], 'answer_start': [30]} 30 35 17 382 0 0
token answer decode: [CLS]
{'text': ['1963年'], 'answer_start': [30]} 30 35 17 289 0 0
token answer decode: [CLS]
{'text': ['1990年被擢升为天主教河内总教区宗座署理'], 'answer_start': [41]} 41 62 15 382 53 70
token answer decode: 1990 年 被 擢 升 为 天 主 教 河 内 总 教 区 宗 座 署 理
{'text': ['1990年被擢升为天主教河内总教区宗座署理'], 'answer_start': [41]} 41 62 15 382 0 0
token answer decode: [CLS]
{'text': ['1990年被擢升为天主教河内总教区宗座署理'], 'answer_start': [41]} 41 62 15 283 0 0
token answer decode: [CLS]
{'text': ['范廷颂于1919年6月15日在越南宁平省天主教发艳教区出生'], 'answer_start': [97]} 97 126 15 382 100 124
token answer decode: 范 廷 颂 于 1919 年 6 月 15 日 在 越 南 宁 平 省 天 主 教 发 艳 教 区 出 生
{'text': ['范廷颂于1919年6月15日在越南宁平省天主教发艳教区出生'], 'answer_start': [97]} 97 126 15 382 0 0
token answer decode: [CLS]
{'text': ['范廷颂于1919年6月15日在越南宁平省天主教发艳教区出生'], 'answer_start': [97]} 97 126 15 283 0 0
token answer decode: [CLS]
{'text': ['19

In [18]:
def process_func(examples):
  sample_examples=tokenizer(
      text=list(examples["question"]),
      text_pair=list(examples["context"]),
      max_length=384,truncation="only_second",
      padding="max_length",
      return_offsets_mapping=True,
      return_overflowing_tokens=True,
      stride=128
      )
  sample_mapping=sample_examples.pop("overflow_to_sample_mapping")
  start_positions=[]
  end_positions=[]
  examples_ids=[]
  for idx,_ in enumerate(sample_mapping):
    answer=examples["answers"][sample_mapping[idx]]
    start_char=answer["answer_start"][0]
    end_char=start_char+len(answer["text"][0])
    # 定位答案在token中的起始位置和结束位置
    # 一种策略，我们要拿到context的起始和结束，然后从左右两侧向答案逼近
    context_start=sample_examples.sequence_ids(idx).index(1)
    context_end=sample_examples.sequence_ids(idx).index(None,context_start)-1
    offset=sample_examples.get("offset_mapping")[idx]
    # 判断答案是否在context中
    if offset[context_end][1]<start_char or offset[context_start][0]>end_char:
      start_token_pos=0
      end_token_pos=0
    else:
      token_id=context_start
      while token_id<=context_end and offset[token_id][0]<start_char:
        token_id +=1
      start_token_pos=token_id
      token_id=context_end
      while token_id>=context_start and offset[token_id][1]>end_char:
        token_id -=1
      end_token_pos=token_id
    start_positions.append(start_token_pos)
    end_positions.append(end_token_pos)
    examples_ids.append(examples["id"][sample_mapping[idx]])
    sample_examples["offset_mapping"][idx]=[
      {o if sample_examples.sequence_ids(idx)[k]==1 else None}
      for k,o in enumerate(sample_examples["offset_mapping"][idx])
    ]
  sample_examples["example_ids"]=example_ids
  sample_examples["start_positions"]=start_positions
  sample_examples["end_positions"]=end_positions
  return sample_examples

In [19]:
tokenized_datasets=datasets.map(process_func,batched=True,remove_columns=datasets["train"].column_names)

Map:   0%|          | 0/10142 [00:00<?, ? examples/s]

ArrowInvalid: Column 4 named example_ids expected length 1827 but got length 0

#Step4 获取模型输出

In [ ]:
import numpy as np
import collections
def get_result(start_logits,end_logits,examples,features):
  predictions={}
  references={}
  example_to_feature=collections.defaultdict(list)
  for idx,example_id in enumerate(features["example_id"]):
    example_to_feature[example_id].append(idx)
  n_best=20
  max_answer_length=30
  for example in examples:
    example_id=example["id"]
    context=example["context"]
    answer=[]
    for feature_ids in example_to_feature[example_id]:
      start_logit=start_logits[feature_idx]
      end_logits=end_logits[feature_idx]
      offset=feature[feature_ids]["offset_mapping"]
      start_indexes=np.argsort(start_logit)[::-1][:n_best].tolist()
      end_indexes=np.argsort(end_logit)[::-1][:n_best].tolist()
      for start_index in start_indexes:
        for end_index in end_indexes:
          if offset[start_index] is None or offset[end_index] is None:
            continue
          if end_index<start_index or end_index-start_index+1>max_answer_length:
            continue
          answer.append({
            "text":context[offset[start_index][0]:offset[end_index][1]]
            "score":start_logit[start_index]+end_logit[end_index]
          })

    if len(answer)>0:
      best_answer=max(answers,key=lambda x:x["score"])
      predictions[example_id]=best_answer["text"]
    else:
      predictions[example_id]=""
    references[example_id]=example["answers"]["text"]
  return predictions,references




#Step5 评估函数

In [ ]:
from cmrc_eval import evaluate_cmrc

def metric(pred):
  start_logits,end_logits=pred[0]
  if start_logits.shape[0]==len(sample_examples["validation"])
    p,r =get_result(start_logits,end_logits,sample_example["validation"])
  else:
    p,r =get_result(start_logits,end_logits,sample_example["test"])
  return evaluate_cmrc(p,r)


#Step6 加载模型

In [18]:
model=AutoModelForQuestionAnswering.from_pretrained("hfl/chinese-macbert-base")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  412MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ign

#Step7 配置TrainingArguments

In [19]:
args=TrainingArguments(
    output_dir="models_for_qa",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="epoch",
    logging_steps=50,
    num_train_epochs=3
)

#Step8 配置Trainer

In [20]:
trainer=Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=DefaultDataCollator(),
    compute_metrics=metric
)

#Step9 模型训练

In [21]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.348787,1.085597


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,1.348787,1.085597
2,0.887465,1.099606
3,0.559536,1.294623


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1902, training_loss=1.0522237735967908, metrics={'train_runtime': 2504.1361, 'train_samples_per_second': 12.15, 'train_steps_per_second': 0.76, 'total_flos': 5962661340337152.0, 'train_loss': 1.0522237735967908, 'epoch': 3.0})

#Step8 模型预测

In [22]:
from transformers import pipeline
pipe=pipeline("text-generation",model=model,tokenizer=tokenizer,device=0)
pipe(question="小M在哪里上班？",context="小M在北京上班。")

AttributeError: 'TextGenerationPipeline' object has no attribute 'assistant_model'

In [ ]:
import torch

print("PyTorch 能否使用 GPU：", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU：", torch.cuda.get_device_name(0))